In [ ]:
!pip install -q roboflow ultralytics

In [ ]:
!pip install roboflow

from google.colab import userdata
from roboflow import Roboflow
rf = Roboflow(api_key=userdata.get("ROBOFLOW_API_KEY"))
project = rf.workspace("g18l5754").project("fish4knowledge-dataset")
version = project.version(1)
dataset = version.download("yolov8")


In [ ]:
print(dataset.location)

In [ ]:
import glob
import os

data_files = glob.glob("/content/**/data.yaml", recursive=True)
model_files = glob.glob("/content/**/best.pt", recursive=True)

print("Найдены data.yaml:")
print(data_files)

print("\nНайдены best.pt:")
print(model_files)

print("\nПапки в /content:")
print(os.listdir("/content"))

In [ ]:
!find /content/fish_results -maxdepth 5 -type f

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
from google.colab import userdata

key = userdata.get("ROBOFLOW_API_KEY")

print("Секрет найден:", key is not None)
print("Длина ключа:", len(key) if key else 0)

In [ ]:
from google.colab import userdata
from roboflow import Roboflow

rf = Roboflow(api_key=userdata.get("ROBOFLOW_API_KEY"))
project = rf.workspace("g18l5754").project("fish4knowledge-dataset")
version = project.version(1)
dataset = version.download("yolov8")

print("Dataset downloaded to:", dataset.location)

In [ ]:
import time
import torch
from pathlib import Path
from ultralytics import YOLO

SAVE_DIR = "/content/drive/MyDrive/Fish4Knowledge_results"
RUN_NAME = "yolov8n_fish4knowledge"

print("GPU:", torch.cuda.get_device_name(0))
print("Dataset:", dataset.location)
print("Results will be saved to:", SAVE_DIR)

model = YOLO("yolov8n.pt")

start_time = time.time()

results = model.train(
    data=f"{dataset.location}/data.yaml",
    epochs=30,
    imgsz=640,
    batch=16,
    device=0,
    seed=42,
    deterministic=True,
    workers=2,
    project=SAVE_DIR,
    name=RUN_NAME,
    exist_ok=True
)

training_time_minutes = (time.time() - start_time) / 60

metadata_file = Path(SAVE_DIR) / RUN_NAME / "experiment_metadata.txt"
metadata_file.write_text(
    f"GPU: {torch.cuda.get_device_name(0)}\n"
    f"Training time: {training_time_minutes:.2f} minutes\n"
    f"Epochs: 30\n"
    f"Image size: 640\n"
    f"Batch size: 16\n"
    f"Random seed: 42\n",
    encoding="utf-8"
)

print(f"Training completed in {training_time_minutes:.2f} minutes")
print("Saved model:", f"{SAVE_DIR}/{RUN_NAME}/weights/best.pt")

In [ ]:
from ultralytics import YOLO
from pathlib import Path

MODEL_PATH = "/content/drive/MyDrive/Fish4Knowledge_results/yolov8n_fish4knowledge/weights/best.pt"
DATA_PATH = "/content/Fish4Knowledge-Dataset-1/data.yaml"
TEST_DIR = "/content/drive/MyDrive/Fish4Knowledge_results"

best_model = YOLO(MODEL_PATH)

test_metrics = best_model.val(
    data=DATA_PATH,
    split="test",
    imgsz=640,
    batch=16,
    device=0,
    plots=True,
    project=TEST_DIR,
    name="test_evaluation",
    exist_ok=True
)

precision = test_metrics.box.mp
recall = test_metrics.box.mr
map50 = test_metrics.box.map50
map5095 = test_metrics.box.map

print("\nFINAL TEST RESULTS")
print(f"Precision: {precision:.4f}")
print(f"Recall: {recall:.4f}")
print(f"mAP@50: {map50:.4f}")
print(f"mAP@50-95: {map5095:.4f}")

summary_path = Path(TEST_DIR) / "test_evaluation" / "test_metrics.txt"
summary_path.write_text(
    f"Dataset: Fish4Knowledge v1, real underwater images\n"
    f"Model: YOLOv8n\n"
    f"Test images: 188\n"
    f"Precision: {precision:.4f}\n"
    f"Recall: {recall:.4f}\n"
    f"mAP@50: {map50:.4f}\n"
    f"mAP@50-95: {map5095:.4f}\n",
    encoding="utf-8"
)

print("Metrics saved to:", summary_path)

In [ ]:
from ultralytics import YOLO
from pathlib import Path

MODEL_PATH = "/content/drive/MyDrive/Fish4Knowledge_results/yolov8n_fish4knowledge/weights/best.pt"
TEST_IMAGES = "/content/Fish4Knowledge-Dataset-1/test/images"
PREDICTION_DIR = "/content/drive/MyDrive/Fish4Knowledge_results"

best_model = YOLO(MODEL_PATH)

prediction_results = best_model.predict(
    source=TEST_IMAGES,
    imgsz=640,
    conf=0.25,
    device=0,
    save=True,
    project=PREDICTION_DIR,
    name="test_predictions",
    exist_ok=True
)

print("Prediction images saved to:")
print("/content/drive/MyDrive/Fish4Knowledge_results/test_predictions")

In [ ]:
import glob
from IPython.display import Image, display

prediction_files = glob.glob(
    "/content/drive/MyDrive/Fish4Knowledge_results/test_predictions/*"
)

print("Saved prediction images:", len(prediction_files))

if prediction_files:
    display(Image(filename=prediction_files[0], width=800))

In [ ]:
import yaml

with open("/content/Fish4Knowledge-Dataset-1/data.yaml", "r", encoding="utf-8") as file:
    dataset_config = yaml.safe_load(file)

print("Number of classes:", dataset_config.get("nc"))
print("Class names:")
print(dataset_config.get("names"))

In [ ]:
!cat /content/Fish4Knowledge-Dataset-1/data.yaml

In [ ]:
from pathlib import Path
import shutil
import yaml

source_dir = Path("/content/Fish4Knowledge-Dataset-1")
target_dir = Path("/content/Fish4Knowledge-OneClass")

if target_dir.exists():
    shutil.rmtree(target_dir)

shutil.copytree(source_dir, target_dir)

label_files = []

for split in ["train", "valid", "test"]:
    labels_dir = target_dir / split / "labels"

    for label_file in labels_dir.glob("*.txt"):
        lines = label_file.read_text(encoding="utf-8").splitlines()
        converted_lines = []

        for line in lines:
            parts = line.strip().split()

            if len(parts) >= 5:
                parts[0] = "0"
                converted_lines.append(" ".join(parts))

        label_file.write_text(
            "\n".join(converted_lines),
            encoding="utf-8"
        )
        label_files.append(label_file)

yaml_path = target_dir / "data.yaml"

with open(yaml_path, "r", encoding="utf-8") as file:
    config = yaml.safe_load(file)

config["nc"] = 1
config["names"] = ["fish"]
config["train"] = str(target_dir / "train" / "images")
config["val"] = str(target_dir / "valid" / "images")
config["test"] = str(target_dir / "test" / "images")

with open(yaml_path, "w", encoding="utf-8") as file:
    yaml.safe_dump(config, file, sort_keys=False)

print("Converted label files:", len(label_files))
print("New dataset:", target_dir)
print("Classes:", config["names"])

In [ ]:
import time
import torch
from pathlib import Path
from ultralytics import YOLO

DATA_PATH = "/content/Fish4Knowledge-OneClass/data.yaml"
SAVE_DIR = "/content/drive/MyDrive/Fish4Knowledge_results"
RUN_NAME = "yolov8n_fish_oneclass"

print("GPU:", torch.cuda.get_device_name(0))
print("Dataset:", DATA_PATH)
print("Results directory:", SAVE_DIR)

model = YOLO("yolov8n.pt")

start_time = time.time()

results = model.train(
    data=DATA_PATH,
    epochs=30,
    imgsz=640,
    batch=16,
    device=0,
    seed=42,
    deterministic=True,
    workers=2,
    project=SAVE_DIR,
    name=RUN_NAME,
    exist_ok=True
)

training_time_minutes = (time.time() - start_time) / 60

metadata_path = Path(SAVE_DIR) / RUN_NAME / "experiment_metadata.txt"

metadata_path.write_text(
    "Dataset: Fish4Knowledge Roboflow derivative v1\n"
    "Task: single-class fish detection\n"
    "Original numeric categories merged into one class: fish\n"
    "Images: 1879\n"
    "Model: YOLOv8n\n"
    "Epochs: 30\n"
    "Image size: 640\n"
    "Batch size: 16\n"
    "Random seed: 42\n"
    f"GPU: {torch.cuda.get_device_name(0)}\n"
    f"Training time: {training_time_minutes:.2f} minutes\n",
    encoding="utf-8"
)

print(f"Training completed in {training_time_minutes:.2f} minutes")
print(
    "Best model:",
    f"{SAVE_DIR}/{RUN_NAME}/weights/best.pt"
)

In [ ]:
import os

model_path = "/content/drive/MyDrive/Fish4Knowledge_results/yolov8n_fish_oneclass/weights/best.pt"

print("Модель сохранена:", os.path.exists(model_path))
print(
    "Размер файла:",
    os.path.getsize(model_path) if os.path.exists(model_path) else 0
)

In [ ]:
from ultralytics import YOLO

model_path = "/content/drive/MyDrive/Fish4Knowledge_results/yolov8n_fish_oneclass/weights/best.pt"
data_path = "/content/Fish4Knowledge-OneClass/data.yaml"

model = YOLO(model_path)

metrics = model.val(
    data=data_path,
    split="test",
    imgsz=640,
    batch=16,
    device=0,
    plots=True,
    project="/content/drive/MyDrive/Fish4Knowledge_results",
    name="test_oneclass",
    exist_ok=True
)

In [ ]:
from pathlib import Path

output_file = Path(
    "/content/drive/MyDrive/Fish4Knowledge_results/"
    "test_oneclass/test_metrics.txt"
)

text = """
Dataset: Fish4Knowledge Roboflow derivative v1
Task: single-class fish detection
Test images: 188
Annotated fish instances: 751
Class: fish
Model: YOLOv8n
Precision: 0.887
Recall: 0.885
mAP@50: 0.934
mAP@50-95: 0.576
Preprocessing time: 2.9 ms/image
Inference time: 4.6 ms/image
Postprocessing time: 2.4 ms/image
GPU: NVIDIA Tesla T4
"""

output_file.write_text(text.strip(), encoding="utf-8")

print("Saved:", output_file)

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

run_dir = Path(
    "/content/drive/MyDrive/Fish4Knowledge_results/"
    "yolov8n_fish_oneclass"
)

csv_path = run_dir / "results.csv"
df = pd.read_csv(csv_path)

df.columns = [column.strip() for column in df.columns]

fig, axes = plt.subplots(2, 2, figsize=(12, 8))

axes[0, 0].plot(df["epoch"], df["train/box_loss"], label="Training")
axes[0, 0].plot(df["epoch"], df["val/box_loss"], label="Validation")
axes[0, 0].set_title("Bounding-box loss")
axes[0, 0].set_xlabel("Epoch")
axes[0, 0].legend()
axes[0, 0].grid(alpha=0.3)

axes[0, 1].plot(
    df["epoch"],
    df["train/cls_loss"],
    label="Training"
)
axes[0, 1].plot(
    df["epoch"],
    df["val/cls_loss"],
    label="Validation"
)
axes[0, 1].set_title("Classification loss")
axes[0, 1].set_xlabel("Epoch")
axes[0, 1].legend()
axes[0, 1].grid(alpha=0.3)

axes[1, 0].plot(
    df["epoch"],
    df["metrics/precision(B)"],
    label="Precision"
)
axes[1, 0].plot(
    df["epoch"],
    df["metrics/recall(B)"],
    label="Recall"
)
axes[1, 0].set_title("Precision and recall")
axes[1, 0].set_xlabel("Epoch")
axes[1, 0].legend()
axes[1, 0].grid(alpha=0.3)

axes[1, 1].plot(
    df["epoch"],
    df["metrics/mAP50(B)"],
    label="mAP@50"
)
axes[1, 1].plot(
    df["epoch"],
    df["metrics/mAP50-95(B)"],
    label="mAP@50–95"
)
axes[1, 1].set_title("Detection performance")
axes[1, 1].set_xlabel("Epoch")
axes[1, 1].legend()
axes[1, 1].grid(alpha=0.3)

plt.tight_layout()

output_path = run_dir / "results.png"
plt.savefig(output_path, dpi=300, bbox_inches="tight")
plt.show()

print("Saved:", output_path)

In [ ]:
from pathlib import Path
import zipfile

base_dir = Path(
    "/content/drive/MyDrive/Fish4Knowledge_results"
)

folders = [
    base_dir / "yolov8n_fish_oneclass",
    base_dir / "test_oneclass"
]

zip_path = Path(
    "/content/drive/MyDrive/"
    "Fish4Knowledge_oneclass_complete.zip"
)

with zipfile.ZipFile(
    zip_path,
    "w",
    compression=zipfile.ZIP_DEFLATED
) as archive:

    for folder in folders:
        for file_path in folder.rglob("*"):
            if file_path.is_file():
                archive.write(
                    file_path,
                    file_path.relative_to(base_dir)
                )

print("ZIP created:", zip_path)